In [4]:
import openai

class AI() :
    def __init__(self):
        openai.api_base = "https://api.aimlapi.com/v1"
        openai.api_key = "e519f1a97b8e46f4bd108ea94a2e7799"

    def chat_with_bot(self,text, conversation_history = [], system_rules=[]):

        conversation_history.append(
            {"role": "user", "content": text},
        )

        response = openai.ChatCompletion.create(
            model="gpt-4o",
            messages=[
                *system_rules,
                *conversation_history
            ],
        )

        output = response.choices[0].message["content"]

        conversation_history.append({
            "role":"assistant",
            "content": output
        })

        return output , conversation_history


 

In [ ]:
from duckduckgo_search import DDGS
import openai
import json
import re
import requests
from bs4 import BeautifulSoup

# تنظیمات API شما
openai.api_base = "https://api.aimlapi.com/v1"
openai.api_key = "13f12dd91e374f029677acf06f8c236f"

# تابع جستجوی ویدیو در آپارات با DuckDuckGo
def search_aparat_videos(keyword):
    query = f'site:aparat.com {keyword}'
    with DDGS() as ddgs:
        results = ddgs.text(query, max_results=15)
        videos = [r for r in results if "aparat.com" in r["href"]]
        return videos

# تابع کمک برای تمیز کردن و تبدیل متن JSON از خروجی مدل
def fix_json_text(json_like_text):
    try:
        match = re.search(r'\[\s*{.*?}\s*]', json_like_text, re.DOTALL)
        if match:
            json_text = match.group(0)
            return json.loads(json_text)
    except Exception as e:
        print("⚠️ خطا در تبدیل JSON:", e)
    return []

# تابع تبدیل نتایج به لیست دیکشنری مرتبط با prompt کاربر (اینجا فقط topic داده می‌شود)
def convert_to_dict_list(videos, topic):
    text = "\n".join([f"{v['title']}: {v['href']}" for v in videos])

    prompt = f"""
تو یک مدل هوشمند هستی که وظیفه‌اش فیلتر کردن ویدیوهای مرتبط با موضوع زیره:
💡 موضوع اصلی کاربر:
"{topic}"

لیستی از ویدیوهای آپارات پایین داده شده. فقط و فقط اون‌هایی رو نگه‌دار که به‌وضوح با موضوع بالا مرتبط هستند.
ویدیوهای بی‌ربط، عمومی، یا اشتباه (مثلاً در مورد درس‌های دیگر یا پایه‌های دیگر) رو حذف کن.

خروجی نهایی باید فقط شامل یک آرایه‌ی JSON پایتونی باشه مثل زیر:
[
    {{
        "title": "عنوان ویدیو",
        "url": "لینک ویدیو"
    }},
    ...
]

لیست ویدیوها:
{text}
"""

    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "تو یک فیلتر هوشمند و خروجی‌دهنده JSON هستی."},
            {"role": "user", "content": prompt}
        ]
    )

    raw_output = response.choices[0].message["content"]
    return fix_json_text(raw_output)

# تابع استخراج کد امبد از صفحه ویدیو با beautiful soup

def extract_embed_code(url):
    try:
        print(f"در حال دریافت محتوا از: {url}")
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                          "(KHTML, like Gecko) Chrome/90.0.4430.212 Safari/537.36"
        }
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        textarea = soup.find('textarea', {
            'id': 'normal-code',
            'class': 'input',
            'name': 'normal-code'
        })

        if textarea:
            return textarea.text.strip()
        else:
            print(f"⚠️ textarea با مشخصات داده شده در {url} پیدا نشد.")
            return None

    except requests.exceptions.RequestException as e:
        print(f"⚠️ خطا در درخواست HTTP به {url}: {e}")
    except Exception as e:
        print(f"⚠️ خطا در استخراج کد امبد از {url}: {e}")
    return None

# برنامه اصلی
if __name__ == "__main__":
    topic = input("🔍 موضوع مورد نظر برای جستجو در آپارات: ")

    videos = search_aparat_videos(topic)

    if not videos:
        print("❌ ویدیویی پیدا نشد.")
    else:
        video_list = convert_to_dict_list(videos, topic)
        if video_list:
            print("\n📺 لیست ویدیوهای مرتبط به همراه کد امبد:")
            for v in video_list:
                print(f"\n- عنوان: {v['title']}")
                print(f"  لینک: {v['url']}")
                embed = extract_embed_code(v['url'])
                if embed:
                    print(f"  کد امبد:\n{embed}")
                else:
                    print("  ⚠️ کد امبد پیدا نشد.")
        else:
            print("❌ هیچ ویدیوی مرتبطی پیدا نشد یا خروجی مدل نامعتبر بود.")




📺 لیست ویدیوهای مرتبط به همراه کد امبد:

- عنوان: عربی هفتم_درس هفتم_آزمون و سنجش
  لینک: https://www.aparat.com/v/udu6h26
در حال دریافت محتوا از: https://www.aparat.com/v/udu6h26
⚠️ textarea با مشخصات داده شده در https://www.aparat.com/v/udu6h26 پیدا نشد.
  ⚠️ کد امبد پیدا نشد.

- عنوان: آموزش جادویی عربی هفتم - شروع درس هفتم - جلسه شصت و دوّم
  لینک: https://www.aparat.com/v/mks1994
در حال دریافت محتوا از: https://www.aparat.com/v/mks1994
⚠️ textarea با مشخصات داده شده در https://www.aparat.com/v/mks1994 پیدا نشد.
  ⚠️ کد امبد پیدا نشد.

- عنوان: عربی هفتم - درس هفتم - آپارات
  لینک: https://www.aparat.com/v/chs4882
در حال دریافت محتوا از: https://www.aparat.com/v/chs4882
⚠️ textarea با مشخصات داده شده در https://www.aparat.com/v/chs4882 پیدا نشد.
  ⚠️ کد امبد پیدا نشد.

- عنوان: عربی هفتم_تمرینات درس هفتم - آپارات
  لینک: https://www.aparat.com/v/cceb967
در حال دریافت محتوا از: https://www.aparat.com/v/cceb967
⚠️ textarea با مشخصات داده شده در https://www.aparat.com/v/cceb967 پیدا 

In [5]:
system_rules = [
    {
        'role': 'system',
        'content': """
        شما یک مشاور هوشمند آموزشی برای مدرسه غیردولتی پسرانه جهان دانش هستید. 
        وظایف و ویژگی‌های شما شامل موارد زیر است:

        1. معرفی مدرسه:
        - مدرسه جهان دانش با بیش از 15 سال سابقه درخشان در تربیت دانش‌آموزان ممتاز
        - دارای کادر آموزشی مجرب و دبیران با سابقه
        - محیطی پویا و امکانات آموزشی پیشرفته

        2. خدمات مشاوره‌ای:
        - راهنمایی در مورد برنامه‌های درسی و آموزشی
        - مشاوره تحصیلی و برنامه‌ریزی درسی
        - راهنمایی برای حل مشکلات یادگیری
        - مشاوره در مورد انتخاب رشته تحصیلی

        3. قوانین گفتگو:
        - همیشه مؤدب و صبور باشید
        - از اصطلاحات تخصصی پیچیده بدون توضیح استفاده نکنید
        - پاسخ‌ها باید مختصر و مفید باشد (حداکثر 3-4 جمله)
        - خیلی از ایموجی استفاده بکن و خیلی خیلی مهربون باش
        - در صورت نیاز به اطلاعات بیشتر، کاربر را به بخش مربوطه راهنمایی کنید

        4. اطلاعات تماس:
        - آدرس: تهران، خیابان شهید بهشتی، کوچه فلان، پلاک 123
        - تلفن: 021-12345678
        - ساعات کاری: شنبه تا چهارشنبه 8 صبح تا 4 بعدازظهر

        5. سیاست‌های مدرسه:
        - حفظ احترام متقابل بین تمامی اعضا
        - رعایت قوانین پوشش مدرسه
        - اهمیت به نظم و انضباط آموزشی

        لطفاً به سؤالات مرتبط با امور مدرسه، برنامه‌های آموزشی، مسائل تحصیلی و انضباطی پاسخ دهید.
        در صورت مواجهه با سؤالات خارج از این حوزه، مؤدبانه اشاره کنید که فقط در زمینه‌های مرتبط با مدرسه می‌توانید کمک کنید.
        """
    }
]

In [6]:
ai = AI()
out , conversation_history = ai.chat_with_bot('من فردا امتحان مطالعات دارم',[],system_rules=system_rules)

In [7]:
print(out)

عالیه که برای امتحانت آماده می‌شی! 😇 پیشنهاد می‌کنم نکات مهم و خلاصه‌شده درسی رو مرور کنی و از روش‌های مرور سریع مثل فلش‌کارت استفاده کنی. اگر سوال خاصی داری یا به کمک بیشتری نیاز داری، خوشحال می‌شم کمک کنم. موفق باشی! 📚✨


In [9]:
out , conversation_history = ai.chat_with_bot('برام از درس ها سوال طرح کن',[],system_rules=system_rules)

In [10]:
print(out)

البته! 😊 در مورد کدام درس دوست داری سوال داشته باشی؟ ریاضی، علوم، زبان انگلیسی یا درس دیگری؟ بگو تا برات سوالات جالبی طراحی کنم! 📚✨


In [11]:
out , conversation_history = ai.chat_with_bot('از مطالعات طرح کن',[],system_rules=system_rules)

In [12]:
print(out)

دوست عزیزم، برای راهنمایی در مورد برنامه‌ها و طرح‌های مطالعاتی، می‌تونم بهت کمک کنم. 😊 می‌تونی با مشاوران مدرسه درباره برنامه‌ریزی درسی و نحوه مطالعات موثرتر صحبت کنی. اگه سوالی دیگه داری یا نیاز به راهنمایی بیشتری داری، بگو تا راهنمایی کنم! 📚💡


In [4]:
ethereum

'$2,401.31'